**How would we translate a query like this?**
``` sql
SELECT max(b) 
FROM stuff
```

**Create some fake data**


In [1]:
from faker import Faker
fake = Faker()

from dataclasses import dataclass, field
import heapq

number_of_tuples = 40

@dataclass
class Stuff:
    a: int
    b: int

# create some fake data mimicking tuples in a relation:
data = [Stuff(fake.pyint(max_value=1000), fake.pyint(max_value=2000)) for f in range(number_of_tuples)]
data[:10]

[Stuff(a=113, b=1038),
 Stuff(a=800, b=1790),
 Stuff(a=426, b=1492),
 Stuff(a=931, b=1875),
 Stuff(a=99, b=1249),
 Stuff(a=55, b=513),
 Stuff(a=14, b=1089),
 Stuff(a=586, b=347),
 Stuff(a=214, b=113),
 Stuff(a=946, b=1923)]

**Algorithm 1: Full-blown Aggregation**

In [2]:
import math
def aggregate(data, attribute):
    current_max = -math.inf
    for el in data:
        current_max = max(current_max, el.__dict__[attribute])
    return current_max

aggregate(data, "b")

1923

**Algorithm 2: Online Aggregation**

In [3]:
import math
def online_aggregation(data, attribute):
    current_max = -math.inf
    for el in data:
        current_max = max(current_max, el.__dict__[attribute])
        yield current_max

for agg in online_aggregation(data, "b"):
    print(agg)

1038
1790
1790
1875
1875
1875
1875
1875
1875
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923
1923


In [4]:
import unittest

class MyTest(unittest.TestCase):

    # the following function will be called before every test-function:

    def create_data(self, number_of_tuples):
        self.data = [Stuff(fake.pyint(max_value=20), fake.pyint(max_value=20)) for f in range(number_of_tuples)]

    def test_scans_end_to_end(self):
        for i in range(10):
            self.create_data(1000)

            res1 = aggregate(self.data, "b")
            *_, res2 = online_aggregation(self.data, "b")

            self.assertEqual(res1,res2)
    
# Run the unit test without shutting down the jupyter kernel
# notice that this call will look for all test classes, i.e.
# all classes inheriting from unittest.TestCase
# each of these functions will be executed once
# unittest.main(argv=['ignored', '-v'], verbosity=2, exit=False)

# only execute a specific Test-class:
unittest.main(argv=['ignored', '-v'], defaultTest='MyTest', verbosity=2, exit=False)


test_scans_end_to_end (__main__.MyTest.test_scans_end_to_end) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.019s

OK
